# Scanpy UMAP and shared 2D embeddings with GenomeSpy

Keep your Scanpy analysis and replace supported UMAP plotting calls with `genome_spy.integrations.scanpy`. Start with basic plotting, then linked brushing and Python retrieval, and finally customization. These examples adapt the [core](https://scanpy.scverse.org/en/stable/tutorials/plotting/core.html) and [advanced](https://scanpy.scverse.org/en/stable/tutorials/plotting/advanced.html) plotting tutorials.

Scanpy’s packaged `pbmc68k_reduced()` contains 700 cells, 765 genes, annotations, and **UMAP coordinates already stored in `obsm["X_umap"]`**. GenomeSpy reads those coordinates. For your own data, run preprocessing, neighbors, and `sc.tl.umap(adata)` explicitly in Scanpy before plotting. Install this project with its `scanpy` extra and select that environment as the notebook kernel. The preview targets Scanpy 1.11.5.

For expression plotting examples, see the separate [expression plotting notebook](scanpy_expression.ipynb).


In [ ]:
import scanpy as sc

import genome_spy.integrations.scanpy as gsc

pbmc = sc.datasets.pbmc68k_reduced()
pbmc

## 1. Basic plotting

The tutorial’s `sc.pl.umap(pbmc, color=["louvain", "CD79A"])` becomes the call below. The categorical annotation and gene expression panels share the stored embedding but keep independent color scales.

`width` and `height` set each GenomeSpy plot area in logical pixels; axes and legends add space. Hover a cell to inspect its barcode and value. Use the mouse wheel or trackpad over a plot to zoom both axes, and drag to pan. Panels within each chart navigate together.


In [ ]:
chart = gsc.pl.umap(
    pbmc,
    color=["louvain", "CD79A"],
    size=20,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

## 2. Link a brush across panels

Add a native interval selection to the same plotting call. **Hold Shift and drag a rectangle in the left plot.** Both plots highlight the same cells and the count updates below them. Double-click in the left plot to clear the brush.

Use the mouse wheel or trackpad to zoom both axes, and ordinary dragging to pan. Both panels share their coordinate view. The right plot displays the linked selection; brushing starts in the left plot.


In [ ]:
import genome_spy as gs

brush = gs.selection_interval("cells", encodings=["x", "y"])
chart = gsc.pl.umap(
    pbmc,
    color=["louvain", "CD79A"],
    selection=brush,
    zoom=True,
    size=20,
    width=280,
    height=280,
    show=False,
)
view = chart.widget(parameter_names=[brush.name], inline=True, controls=False)
view

### Retrieve the selected cells in Python

After brushing, run the next cell. The normal Jupyter widget synchronizes rectangle bounds into `view.parameter_values`. `selection_mask` matches them to the stored UMAP coordinates in observation order; barcodes come from `obs_names`.

Run this cell again after changing the brush to refresh the subset. Keep the plotted coordinates unchanged. Highlighting and the count also work in exported interactive HTML; Python retrieval requires a live notebook kernel.


In [ ]:
mask = gsc.selection_mask(pbmc, view.parameter_values.get(brush.name))
selected_ids = pbmc.obs_names[mask]
selected_adata = pbmc[mask].copy()
selected_adata

## 3. Customize the plots

Keep the same dataset and embedding. The following examples cover subsetting, titles, frames, category order, CSS palettes, continuous color bounds, and panel layout. Colors use GenomeSpy schemes and opaque CSS colors; no Matplotlib setup is needed.

### Plot a subset, with a title and no frame

The tutorial’s Louvain groups 0, 3, and 9 are colored by S-phase score. This AnnData view retains its existing coordinates; plotting does not compute a new UMAP.


In [ ]:
subset = pbmc[pbmc.obs["louvain"].isin(["0", "3", "9"])]
chart = gsc.pl.umap(
    subset,
    color="S_score",
    title="S-phase score in selected groups",
    frameon=False,
    size=20,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

### Set categorical order and colors

Create an ordered annotation explicitly in Python, as in the tutorial. Legend order follows its categorical levels. The same CSS mapping gives each phase a consistent color in both panels.


In [ ]:
pbmc.obs["phase_ordered"] = pbmc.obs["phase"].cat.reorder_categories(
    ["G1", "S", "G2M"], ordered=True
)
chart = gsc.pl.umap(
    pbmc,
    color=["phase", "phase_ordered"],
    title=["Original order", "Ordered phases"],
    palette={"S": "#17becf", "G1": "#bcbd22", "G2M": "#d62728"},
    s=20,
    wspace=0.5,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

### Choose marker shapes

Use `marker="s"` for squares and `marker="D"` for diamonds. A list chooses a shape per panel. Here both use `size=20` in points squared; Scanpy’s diamond is a rotated square, so its horizontal and vertical spans are larger. GenomeSpy uses native shapes with the matching size conversion. Coordinates, colors, hover and zoom are shared with the other examples. These shapes also work with outlines, stored neighbor edges and linked selection. Category legend symbols follow each panel’s marker shape.

In [ ]:
marker_chart = gsc.pl.umap(
    pbmc,
    color=["louvain", "CD79A"],
    marker=["s", "D"],
    size=20,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
marker_chart.display(inline=True, controls=False)

### Outline cell clusters

Use `add_outline=True` to draw Scanpy-style black and white outlines behind the same stored UMAP points. The colored foreground defaults to opacity 0.7; set `alpha` to change it. `outline_width=(0.3, 0.05)` sets the relative outer and inner widths; `outline_color=("black", "white")` supplies GenomeSpy-supported CSS colors. Hover and x/y zoom still work. All shared embedding functions use this renderer; no analysis is run by the plotting call.


In [ ]:
outline_chart = gsc.pl.umap(
    pbmc,
    color=["louvain", "CD79A"],
    add_outline=True,
    size=20,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
outline_chart.display(inline=True, controls=False)

### Show the stored neighbor graph

This packaged dataset already has a Scanpy neighbor graph in `obsp["connectivities"]`. `edges=True` draws its undirected connections beneath the same stored UMAP points. Neither neighbors nor UMAP are recomputed.

The panels share the cell table and edge list. Native lookups attach the stored coordinates to each edge in the browser; hover still describes cells, and zoom keeps the connections aligned. Smaller markers leave room to see the graph.


In [ ]:
edges_chart = gsc.pl.umap(
    pbmc,
    color=["louvain", "CD79A"],
    edges=True,
    edges_width=0.3,
    size=15,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
edges_chart.display(inline=True, controls=False)

### Use a symmetric continuous color range

The advanced tutorial scores the B-cell markers CD79A and MS4A1 with Scanpy. This explicit analysis adds `B_cell_score` to `obs`; the plotting call only reads it. Use native `redblue` with symmetric numeric bounds.


In [ ]:
sc.tl.score_genes(pbmc, ["CD79A", "MS4A1"], score_name="B_cell_score", random_state=0)
maxabs = abs(pbmc.obs["B_cell_score"]).max()
chart = gsc.pl.umap(
    pbmc,
    color="B_cell_score",
    cmap="redblue",
    s=20,
    vmin=-maxabs,
    vmax=maxabs,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

### Put zero at the color midpoint

`vcenter=0` gives negative and positive B-cell scores half of the native color scheme each, while keeping their observed limits. The two panels use the same stored scores and UMAP coordinates: a linear color scale on the left, and zero at the color midpoint on the right.

Centers also accept percentile strings or functions, just like `vmin` and `vmax`. A list sets a different center for each panel; `None` keeps its linear scale. Explicit bounds must fall below and above the center.


In [ ]:
centered_chart = gsc.pl.umap(
    pbmc,
    color=["B_cell_score", "B_cell_score"],
    vcenter=[None, 0],
    cmap="redblue",
    title=["Linear color scale", "Zero at the color midpoint"],
    size=20,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
centered_chart.display(inline=True, controls=False)

### Compute color bounds per panel

Use functions for `vmin` and `vmax` when each panel needs its own limits. Here the lower bound is the minimum and the upper bound is the 99th percentile of each gene's expression.

The functions run once in Python when the chart is created. They receive the panel's masked color values in painting order, including missing values, so use NumPy's `nan` functions when needed. Zooming keeps these color limits fixed.


In [ ]:
import numpy as np

bounds_chart = gsc.pl.umap(
    pbmc,
    color=["CD79A", "MS4A1"],
    vmin=np.nanmin,
    vmax=lambda values: np.nanpercentile(values, 99),
    size=20,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
bounds_chart.display(inline=True, controls=False)

### Arrange the tutorial’s marker overview

Keep the core tutorial’s marker list, frame setting, four-column layout and per-panel 99th-percentile upper color limit. Use smaller markers (`s=20`) to keep the clusters readable. The biological labels and `n_counts` come from the packaged dataset. Use horizontal scrolling if the overview is wider than the notebook.

In [ ]:
color_vars = [
    "CD79A",
    "MS4A1",
    "IGJ",
    "CD3D",
    "FCER1A",
    "FCGR3A",
    "n_counts",
    "bulk_labels",
]
chart = gsc.pl.umap(
    pbmc,
    color=color_vars,
    s=20,
    frameon=False,
    ncols=4,
    vmax="p99",
    width=240,
    height=240,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

## 4. Reuse the same plotting API for other 2D embeddings

PCA coordinates and their explained variance are already packaged with this dataset. `gsc.pl.pca` uses the same data, colors, points, hover, zoom and selection helpers as UMAP. `annotate_var_explained=True` reads the saved variance ratios; it does not fit PCA. `gsc.pl.pca_scatter` is the same function, matching Scanpy’s alias.

For an arbitrary stored basis, use `gsc.pl.embedding(pbmc, "pca", ...)`; `dimensions=(1, 2)` chooses the second and third columns. Direct obsm names take precedence over the `X_` form. When retrieving a selection from another basis, pass that name too: `gsc.selection_mask(pbmc, snapshot, basis="pca")`.


In [ ]:
pca_chart = gsc.pl.pca(
    pbmc,
    color=["louvain", "CD79A"],
    annotate_var_explained=True,
    size=20,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
pca_chart.display(inline=True, controls=False)

### Inspect stored PCA loadings

The packaged `varm["PCs"]` has loadings for the 309 highly variable genes used in
PCA; other rows are missing. Select those genes explicitly before plotting.
The next call ranks the highest and lowest loadings for the first two components.
The gap marks omitted ranks. Hover a gene label for its loading score.
This uses the same rank panels as `gsc.pl.ranking`; no PCA is computed.


In [ ]:
pca_genes = pbmc[:, pbmc.var["highly_variable"]]
loadings_chart = gsc.pl.pca_loadings(
    pca_genes,
    components="1,2",
    n_points=20,
    width=280,
    height=280,
    show=False,
)
loadings_chart.display(inline=True, controls=False)

### Inspect stored PCA variance ratios

The packaged `uns["pca"]["variance_ratio"]` contains the explained variance for
50 PCs. Plot those stored values with the same ranking helpers; no PCA is run.
`log=True` takes the natural logarithm of scores before ranking, rather than
using a logarithmic axis. Original PC numbers stay attached to their scores.
Scanpy includes the next PC when available, so `n_pcs=10` displays eleven labels.
Hover a label for its plotted score and rank.


In [ ]:
variance_chart = gsc.pl.pca_variance_ratio(
    pbmc,
    n_pcs=10,
    log=True,
    width=360,
    height=240,
    show=False,
)
variance_chart.display(inline=True, controls=False)

### Show the complete PCA overview

`pca_overview` combines PCA scatter, the first three loading panels and variance
ratios in one native chart. It reuses the three plotting functions above and reads
the same stored results. Use the explicit highly-variable-gene subset so its
loadings are finite; plotting never fits PCA or drops genes silently.

As in Scanpy, scatter options affect only the embedding row. Here `zoom=True`
enables x/y navigation there, while the diagnostics keep their default ranks,
components and dimensions. Hover the points or diagnostic labels for their values.


In [ ]:
overview_chart = gsc.pl.pca_overview(
    pca_genes,
    color="louvain",
    size=20,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
overview_chart.display(inline=True, controls=False)

### Prepare another embedding explicitly with Scanpy

The packaged object has no t-SNE coordinates. This preparation cell computes them with Scanpy using the saved PCA representation and a fixed seed. The plotting cell only reads `X_tsne`. Re-running a plotting call never re-runs t-SNE, PCA or neighbors.

`gsc.pl.diffmap` similarly reads stored diffusion coordinates, and `gsc.pl.draw_graph` reads a saved graph layout. Their analysis remains an explicit Scanpy step; missing coordinates raise an error.


In [ ]:
sc.tl.tsne(pbmc, use_rep="X_pca", random_state=0, n_jobs=1)

In [ ]:
tsne_chart = gsc.pl.tsne(
    pbmc,
    color=["louvain", "CD79A"],
    size=20,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
tsne_chart.display(inline=True, controls=False)

## 5. Scatter annotations or expression without an embedding

`scatter` can plot numeric observation annotations directly. Here each point is a cell, with the packaged total counts and detected genes on the axes. The two panels show Louvain groups and mitochondrial percentage; neither plotting call computes these annotations or an embedding. Hover for cell labels and values; wheel to zoom both axes and drag to pan.

Expression keys work too: `gsc.pl.scatter(pbmc, "CD79A", "MS4A1", use_raw=True, show=False)`. To use separate resident layers for x, y and color, pass `layers=("counts", "X", "normalized")` and `use_raw=False` after preparing those layers explicitly. The packaged dataset does not include those layers.

Scanpy uses oversized legend symbols by default. The native configuration below uses compact symbols, six category columns and collected bottom guides while keeping the color scales independent. `height` controls each plot area; axes and outside legends add to the total height.


In [ ]:
scatter_chart = (
    gsc.pl.scatter(
        pbmc,
        x="n_counts",
        y="n_genes",
        color=["louvain", "percent_mito"],
        size=20,
        width=280,
        height=280,
        zoom=True,
        show=False,
    )
    .resolve_legend(default="collected")
    .configure_legend(
        orient="bottom",
        direction="horizontal",
        columns=6,
        symbolSize=64,
        gradientLength=140,
        gradientThickness=10,
        tickCount=3,
        layout={"anchor": "middle", "wrap": True},
    )
)
scatter_chart.display(inline=True, controls=False)

### Plot genes instead of cells

When x, y and color refer to `var` annotations, each point is a gene. These means and dispersions were prepared in the packaged Scanpy dataset. The adapter reads the annotations directly and keeps the gene names in hover; it does not transpose or load the expression matrix.

For stored embeddings, `scatter(basis="umap")` reads `X_umap`. Its older Scanpy API differs from `embedding`: diffusion maps skip the first stored component, and `components` accepts one one-based pair. The named embedding functions above remain the clearer choice for embedding workflows.


In [ ]:
gene_scatter_chart = gsc.pl.scatter(
    pbmc,
    x="means",
    y="dispersions",
    color="dispersions_norm",
    use_raw=False,
    size=20,
    width=360,
    height=280,
    zoom=True,
    show=False,
)
gene_scatter_chart.display(inline=True, controls=False)

## 6. Plot densities computed by Scanpy

Scanpy computes density from the existing UMAP coordinates in the next cell.
`groupby="phase"` scales density within each cell-cycle phase. This does not
recompute UMAP or neighbors. The plotting call only reads the stored density
values and their metadata.


In [ ]:
sc.tl.embedding_density(pbmc, basis="umap", groupby="phase")

Each panel shows one phase in color and the other cells in gray. Density uses
Scanpy's default yellow-to-red scheme and 0–1 bounds. Smaller points keep this
dataset readable. Hover reports the original stored density, including for gray
background cells; zoom and pan stay linked across the panels.


In [ ]:
density_chart = gsc.pl.embedding_density(
    pbmc,
    groupby="phase",
    bg_dotsize=8,
    fg_dotsize=20,
    width=150,
    height=240,
    wspace=0.05,
    zoom=True,
    show=False,
)
density_chart.display(inline=True, controls=False)

**Data use and provenance**

`scanpy.datasets.pbmc68k_reduced()` loads Scanpy’s packaged, processed subset of 10x Genomics PBMC68k. No dataset copy is bundled here. Category ordering, gene scoring and density estimation above are explicit notebook operations; the original UMAP coordinates are retained throughout.

The shared-embedding examples reuse the packaged PCA coordinates, variance ratios and selected gene loadings. A separate explicit Scanpy step computes t-SNE from the packaged PCA representation with random_state=0; plotting only renders those saved coordinates.

The scatter examples read packaged cell and gene annotations; no new statistics are computed.
